# 🍫 Chocolate Sales Analysis
## Complete Data Cleaning, EDA & Machine Learning Pipeline

**Author:** id-drias  
**Dataset:** Chocolate Sales Data (2022-2024)  
**GitHub:** [Repository](https://github.com/id-drias/Chocolate-Sales-Data---Cleaning-Preprocessing)

---

### Overview
This notebook performs a comprehensive analysis of chocolate sales data including:
1. **Data Cleaning & Preprocessing**
2. **Exploratory Data Analysis (EDA)**
3. **Machine Learning Models** (Sales Prediction & Product Classification)
4. **Business Insights & Recommendations**

### Key Findings
- **Total Revenue:** $19.8M | **CAGR:** 6.1%
- **Top Country:** Australia ($3.65M)
- **Top Product:** Peanut Butter Cubes
- **Best ML Model:** Random Forest (R² = 0.54)

## 1. Setup & Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Libraries loaded successfully!')

In [ ]:
# Load data - Kaggle path
# Update this path based on your Kaggle dataset name
df = pd.read_csv('/kaggle/input/chocolate-sales-data/Chocolate Sales (2).csv')

# If running locally, use:
# df = pd.read_csv('Chocolate Sales (2).csv')

print(f'Dataset Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 2. Data Cleaning & Preprocessing

In [ ]:
# Check data quality
print('=== Data Quality Check ===')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nMissing Values:\n{df.isnull().sum()}')
print(f'\nDuplicates: {df.duplicated().sum()}')

In [ ]:
# Clean Amount column: remove $ and commas, convert to float
df['Amount'] = df['Amount'].replace('[\$,]', '', regex=True).astype(float)
print('✓ Amount column cleaned')

# Parse Date column (format: DD/MM/YYYY)
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
print('✓ Date column parsed')

# Extract date components
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.month_name()
df['Day'] = df['Date'].dt.day
df['Weekday'] = df['Date'].dt.day_name()
df['Quarter'] = df['Date'].dt.quarter
df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)
print('✓ Date components extracted')

# Strip whitespace from string columns
for col in ['Sales Person', 'Country', 'Product']:
    df[col] = df[col].str.strip()
print('✓ String columns cleaned')

print(f'\nFinal Shape: {df.shape}')
df.head()

In [ ]:
# Data Summary
print('=== Cleaned Data Summary ===')
print(f"Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Total Revenue: ${df['Amount'].sum():,.0f}")
print(f"Total Boxes: {df['Boxes Shipped'].sum():,}")
print(f"Transactions: {len(df):,}")
print(f"\nUnique Values:")
print(f"  Countries: {df['Country'].nunique()} - {list(df['Country'].unique())}")
print(f"  Products: {df['Product'].nunique()}")
print(f"  Sales People: {df['Sales Person'].nunique()}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Create comprehensive EDA visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Monthly Revenue Trend
monthly = df.groupby('YearMonth')['Amount'].sum().reset_index()
axes[0, 0].plot(range(len(monthly)), monthly['Amount']/1000, marker='o', linewidth=2, markersize=4)
axes[0, 0].set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Revenue ($K)')
axes[0, 0].set_xlabel('Month Index')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Revenue by Country
country_rev = df.groupby('Country')['Amount'].sum().sort_values(ascending=True)
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(country_rev)))
axes[0, 1].barh(country_rev.index, country_rev.values/1e6, color=colors)
axes[0, 1].set_title('Revenue by Country', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Revenue ($M)')

# 3. Top 10 Products
product_rev = df.groupby('Product')['Amount'].sum().sort_values(ascending=False).head(10)
axes[0, 2].barh(product_rev.index, product_rev.values/1e6, color='steelblue')
axes[0, 2].set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
axes[0, 2].set_xlabel('Revenue ($M)')
axes[0, 2].invert_yaxis()

# 4. Yearly Revenue
yearly = df.groupby('Year')['Amount'].sum()
bars = axes[1, 0].bar(yearly.index.astype(str), yearly.values/1e6, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[1, 0].set_title('Yearly Revenue', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Revenue ($M)')
for bar, val in zip(bars, yearly.values):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                    f'${val/1e6:.1f}M', ha='center', fontsize=11)

# 5. Quarterly Pattern
quarterly = df.groupby('Quarter')['Amount'].mean()
axes[1, 1].bar(['Q1', 'Q2', 'Q3', 'Q4'][:len(quarterly)], quarterly.values, 
               color=['#27ae60', '#f39c12', '#e74c3c', '#3498db'][:len(quarterly)])
axes[1, 1].set_title('Average Revenue by Quarter', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Avg Revenue ($)')

# 6. Day of Week Pattern
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_rev = df.groupby('Weekday')['Amount'].sum().reindex(weekday_order)
axes[1, 2].bar(range(7), weekday_rev.values/1e6, color='coral')
axes[1, 2].set_xticks(range(7))
axes[1, 2].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
axes[1, 2].set_title('Revenue by Day of Week', fontsize=14, fontweight='bold')
axes[1, 2].set_ylabel('Revenue ($M)')

plt.tight_layout()
plt.savefig('eda_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA Dashboard saved!')

In [ ]:
# Top Salespeople Analysis
sales_person_stats = df.groupby('Sales Person').agg({
    'Amount': 'sum',
    'Boxes Shipped': 'sum',
    'Date': 'count'
}).rename(columns={'Date': 'Transactions'})
sales_person_stats = sales_person_stats.sort_values('Amount', ascending=False)

print('=== Top 10 Sales People ===')
top10 = sales_person_stats.head(10)
for i, (name, row) in enumerate(top10.iterrows(), 1):
    print(f"{i:2}. {name:20} ${row['Amount']:>12,.0f}  ({row['Transactions']} transactions)")

## 4. Machine Learning - Sales Prediction

In [ ]:
# Feature Engineering
le_country = LabelEncoder()
le_product = LabelEncoder()
le_salesperson = LabelEncoder()
le_weekday = LabelEncoder()

df_ml = df.copy()
df_ml['Country_Encoded'] = le_country.fit_transform(df_ml['Country'])
df_ml['Product_Encoded'] = le_product.fit_transform(df_ml['Product'])
df_ml['SalesPerson_Encoded'] = le_salesperson.fit_transform(df_ml['Sales Person'])
df_ml['Weekday_Encoded'] = le_weekday.fit_transform(df_ml['Weekday'])

# Define features
features = ['Country_Encoded', 'Product_Encoded', 'SalesPerson_Encoded', 
            'Month', 'Quarter', 'Weekday_Encoded', 'Year', 'Boxes Shipped']

X = df_ml[features]
y = df_ml['Amount']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples: {len(X_train)}')
print(f'Testing samples: {len(X_test)}')
print(f'Features: {features}')

In [ ]:
# Train Multiple Regression Models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []
print('=== Training Regression Models ===')
print('-' * 65)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2})
    print(f'{name:20} | R²: {r2:.4f} | RMSE: ${rmse:,.0f} | MAE: ${mae:,.0f}')

results_df = pd.DataFrame(results)
print('-' * 65)
print(f"\n🏆 Best Model: {results_df.loc[results_df['R2'].idxmax(), 'Model']}")

In [ ]:
# Visualize Regression Results
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Feature Importance
rf_model = models['Random Forest']
feat_imp = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

axes[0, 0].barh(feat_imp['Feature'], feat_imp['Importance'], color='steelblue')
axes[0, 0].set_title('Feature Importance (Random Forest)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Importance')

# Actual vs Predicted
y_pred_best = rf_model.predict(X_test)
axes[0, 1].scatter(y_test, y_pred_best, alpha=0.5, c='steelblue', s=20)
axes[0, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 1].set_xlabel('Actual Amount ($)')
axes[0, 1].set_ylabel('Predicted Amount ($)')
axes[0, 1].set_title(f'Actual vs Predicted (R² = {r2_score(y_test, y_pred_best):.3f})', fontsize=12, fontweight='bold')

# Residuals Distribution
residuals = y_test - y_pred_best
axes[1, 0].hist(residuals, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(x=0, color='red', linestyle='--', lw=2)
axes[1, 0].set_xlabel('Residual (Actual - Predicted)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Residuals Distribution', fontsize=12, fontweight='bold')

# Model Comparison
colors = ['#e74c3c' if r < 0.3 else '#f39c12' if r < 0.5 else '#27ae60' for r in results_df['R2']]
axes[1, 1].barh(results_df['Model'], results_df['R2'], color=colors)
axes[1, 1].set_xlabel('R² Score')
axes[1, 1].set_title('Model Comparison', fontsize=12, fontweight='bold')
axes[1, 1].axvline(x=0.5, color='green', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('ml_regression_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Machine Learning - Product Classification

In [ ]:
# Create Product Performance Dataset
product_stats = df.groupby('Product').agg({
    'Amount': ['sum', 'mean', 'count'],
    'Boxes Shipped': ['sum', 'mean']
}).reset_index()
product_stats.columns = ['Product', 'Total_Revenue', 'Avg_Revenue', 'Transactions', 
                         'Total_Boxes', 'Avg_Boxes']
product_stats['Revenue_Per_Box'] = product_stats['Total_Revenue'] / product_stats['Total_Boxes']

# Create performance tiers
q33 = product_stats['Total_Revenue'].quantile(0.33)
q66 = product_stats['Total_Revenue'].quantile(0.66)

def assign_tier(revenue):
    if revenue >= q66: return 'High'
    elif revenue >= q33: return 'Medium'
    else: return 'Low'

product_stats['Performance_Tier'] = product_stats['Total_Revenue'].apply(assign_tier)

print('=== Product Performance Tiers ===')
print(product_stats['Performance_Tier'].value_counts())
print(f"\nThresholds: Low < ${q33:,.0f} | Medium ${q33:,.0f}-${q66:,.0f} | High > ${q66:,.0f}")

In [ ]:
# Train Classification Models
X_class = product_stats[['Avg_Revenue', 'Transactions', 'Avg_Boxes', 'Revenue_Per_Box']]
y_class = product_stats['Performance_Tier']

le_tier = LabelEncoder()
y_class_encoded = le_tier.fit_transform(y_class)

scaler = StandardScaler()
X_class_scaled = scaler.fit_transform(X_class)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class_scaled, y_class_encoded, test_size=0.3, random_state=42
)

classifiers = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

print('=== Classification Results ===')
print('-' * 45)
for name, clf in classifiers.items():
    clf.fit(X_train_c, y_train_c)
    y_pred_c = clf.predict(X_test_c)
    acc = accuracy_score(y_test_c, y_pred_c)
    print(f'{name:25} | Accuracy: {acc:.2%}')
print('-' * 45)

In [ ]:
# Visualize Classification Results
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion Matrix
best_clf = classifiers['Random Forest']
y_pred_final = best_clf.predict(X_test_c)
cm = confusion_matrix(y_test_c, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le_tier.classes_, yticklabels=le_tier.classes_, ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix - Product Classification', fontweight='bold')

# Feature Importance
clf_feat_imp = pd.DataFrame({
    'Feature': ['Avg_Revenue', 'Transactions', 'Avg_Boxes', 'Revenue_Per_Box'],
    'Importance': best_clf.feature_importances_
}).sort_values('Importance', ascending=True)

axes[1].barh(clf_feat_imp['Feature'], clf_feat_imp['Importance'], color='coral')
axes[1].set_xlabel('Importance')
axes[1].set_title('Feature Importance for Classification', fontweight='bold')

plt.tight_layout()
plt.savefig('ml_classification_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Key Insights & Recommendations

In [ ]:
# Summary Statistics
print('=' * 70)
print('                    ANALYSIS SUMMARY')
print('=' * 70)

print('\n📊 KEY METRICS')
print('-' * 40)
print(f"Total Revenue:        ${df['Amount'].sum():>15,.0f}")
print(f"Total Boxes Shipped:  {df['Boxes Shipped'].sum():>15,}")
print(f"Total Transactions:   {len(df):>15,}")
print(f"Avg Transaction:      ${df['Amount'].mean():>15,.2f}")

# Year over Year Growth
yearly_rev = df.groupby('Year')['Amount'].sum()
if len(yearly_rev) > 1:
    yoy_growth = (yearly_rev.iloc[-1] - yearly_rev.iloc[-2]) / yearly_rev.iloc[-2] * 100
    cagr = ((yearly_rev.iloc[-1] / yearly_rev.iloc[0]) ** (1/(len(yearly_rev)-1)) - 1) * 100
    print(f"\nYear-over-Year Growth: {yoy_growth:>14.1f}%")
    print(f"CAGR:                  {cagr:>14.1f}%")

print('\n🌍 TOP PERFORMERS')
print('-' * 40)
top_country = df.groupby('Country')['Amount'].sum().idxmax()
top_product = df.groupby('Product')['Amount'].sum().idxmax()
top_salesperson = df.groupby('Sales Person')['Amount'].sum().idxmax()
print(f"Top Country:          {top_country}")
print(f"Top Product:          {top_product}")
print(f"Top Salesperson:      {top_salesperson}")

print('\n🤖 ML MODEL PERFORMANCE')
print('-' * 40)
best_reg = results_df.loc[results_df['R2'].idxmax()]
print(f"Best Regression:      {best_reg['Model']} (R² = {best_reg['R2']:.4f})")
print(f"Top Predictor:        {feat_imp.iloc[-1]['Feature']}")

print('\n💡 RECOMMENDATIONS')
print('-' * 40)
print('1. Launch Q3 promotional campaigns (seasonal dip)')
print('2. Replicate top performers strategies')
print('3. Expand best products to underperforming markets')
print('4. Optimize Thursday/Friday operations')

print('\n' + '=' * 70)
print('                    ANALYSIS COMPLETE!')
print('=' * 70)

---

## 📁 Files Generated
- `eda_dashboard.png` - EDA visualizations
- `ml_regression_results.png` - Regression analysis
- `ml_classification_results.png` - Classification analysis

## 🔗 Links
- **GitHub:** [Repository](https://github.com/id-drias/Chocolate-Sales-Data---Cleaning-Preprocessing)

## 📝 License
MIT License - Feel free to use and modify!

---
**If you found this notebook helpful, please upvote! 👍**